<a href="https://colab.research.google.com/github/Mohamed-Mohamed-Ibrahim/Code-Generation-and-Guarding/blob/SFT/sft/peft-sft-3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SFT

## Loading libraries

In [ ]:
!apt install tree

In [ ]:
!pip install peft transformers[torch] trl bitsandbytes datasets wandb -U --q

In [ ]:
import os
import random, math
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training

In [ ]:
dataset = load_dataset("flytech/python-codes-25k",split="train")

In [ ]:
dataset

In [ ]:
shuffled_dataset = dataset.shuffle(seed=42)
split_ds_1 = shuffled_dataset.train_test_split(test_size=0.1)
split_ds_2 = split_ds_1['test'].train_test_split(test_size=0.5)

train_dataset = split_ds_1["train"].select(range(100))
eval_dataset = split_ds_2["train"].select(range(100))
test_dataset = split_ds_2["test"].select(range(10))

In [ ]:
print("="*100)
print(shuffled_dataset)
print("="*100)
print(shuffled_dataset[0]['input'])
print("="*100)
print(shuffled_dataset[0]['instruction'])
print("="*100)
print(shuffled_dataset[0]['text'])
print("="*100)
print(shuffled_dataset[0]['output'])
print("="*100)


# Train

In [ ]:
# HF hub ID
BASE_MODEL_NAME = "arnir0/Tiny-LLM"
BASE_MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"

# Push adapter artifacts post training
OUTPUT_DIR = "./qlora-peft-output"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "adapter")

## Hyperparameters

In [ ]:
# Lab (Mandatory from Project Document)
lora_r = 16
lora_target_modules = "all-linear"
bs = 1
gradient_accumulation_steps = 4
epochs = 1
lr = 2e-4
optim = "paged_adamw_8bit"
max_length = 1024

# Others (Recommended / Required for successful run on T4)
lora_alpha = 32                # Standard practice: alpha = 2 * r
lora_dropout = 0.05            # Standard to prevent overfitting
lr_scheduler_type = "cosine"   # Best convergence for LLMs
warmup_ratio = 0.03            # 3% warmup for stability
weight_decay = 0.01


## Wandb

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
WANDB_API_KEY = user_secrets.get_secret("WANDB_API_KEY")

In [ ]:
import wandb
wandb.login(key=WANDB_API_KEY)

In [ ]:
config_dict = {
    "base_model": BASE_MODEL_NAME,
    "lora_r": lora_r,
    "lora_target_modules": lora_target_modules,
    "batch_size": bs,
    "gradient_accumulation_steps": gradient_accumulation_steps,
    "epochs": epochs,
    "lr": lr,
    "optim": optim,
    "max_length": max_length,

    "lora_alpha": lora_alpha,
    "lora_dropout": lora_dropout,
    "scheduler_type": lr_scheduler_type,
    "warmup_ratio": warmup_ratio,
    "quantization": "4bit-nf4",
    "weight_decay": weight_decay
}

# Dynamic run name using the variable
# Example: Qwen/Qwen2-1.5B-Instruct-lora16-bs1-lr0.0002
run_name = f"{BASE_MODEL_NAME.split('/')[-1]}-lora{lora_r}-bs{bs}-lr{lr}"

wandb.init(
    project="sft-nlp-4",
    config=config_dict,
    name=run_name,
    tags=["SFT", "Part-II", BASE_MODEL_NAME.split('/')[-1]]
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    # device_map={"": torch.cuda.current_device()},
    trust_remote_code=True,
    # use_cache=False,
)

# 2. Critical for Qwen/Llama: Ensures dense layers are optimized for training
# model.config.pretraining_tp = 1

model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    target_modules=lora_target_modules,  # Llama-style
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
from trl import SFTConfig, SFTTrainer

# Define ALL configuration parameters within SFTConfig
config = SFTConfig(
    # --- Standard TrainingArguments parameters ---
    output_dir=OUTPUT_DIR,
    learning_rate=lr,
    num_train_epochs=epochs,
    per_device_train_batch_size=bs,
    gradient_accumulation_steps=gradient_accumulation_steps,
    save_strategy="epoch",
    optim=optim,

    weight_decay=weight_decay,

    # --- SFT-specific parameters (moved here from previous iterations) ---
    dataset_text_field="text",
    max_length=max_length,
    packing=False,

    # Scheduler
    lr_scheduler_type=lr_scheduler_type,
    warmup_ratio=warmup_ratio,

    # Compute loss on model answers only
    completion_only_loss=True,

    # Logging
    logging_steps=50,
    report_to="wandb", # enables logging to W&B 😎

    bf16=True,
)

# Initialize the SFTTrainer, passing the SFTConfig object to the 'args' parameter
trainer = SFTTrainer(
    model=model,
    args=config,                # SFTConfig object
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
)

In [ ]:
# from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM

# # 1. Setup the Collator (Critical for "Completion Only" requirement)
# # This tells the model: "Read the Instruction, but only learn from the Response"
# response_template = "### Response:"
# collator = DataCollatorForCompletionOnlyLM(
#     response_template=response_template,
#     tokenizer=tokenizer
# )

# # 2. Define Configuration
# config = SFTConfig(
#     # --- Standard TrainingArguments parameters ---
#     output_dir=OUTPUT_DIR,
#     learning_rate=lr,
#     num_train_epochs=epochs,
#     per_device_train_batch_size=bs,
#     gradient_accumulation_steps=gradient_accumulation_steps,
#     save_strategy="epoch",



#     optim=optim,

#     # --- SFT-specific parameters ---
#     # Note: max_length is often deprecated in favor of max_seq_length in Config
#     max_seq_length=max_length,
#     packing=False,

#     # Scheduler
#     lr_scheduler_type="cosine",
#     warmup_ratio=0.03, # Reverted to 0.03 (Recommended) from 0.1

#     # Logging
#     logging_steps=25,  # More frequent logging is better for short runs
#     report_to="wandb",

#     # Note: We REMOVED 'completion_only_loss=True' here.
#     # We use the 'collator' below instead. It is much safer.
#     dataset_text_field="text", # Ensure your formatting function creates this column
# )

# # 3. Initialize the SFTTrainer
# trainer = SFTTrainer(
#     model=model,
#     args=config,
#     train_dataset=train_dataset,
#     # eval_dataset=eval_dataset, # Optional: SFT often skips eval to save time
#     peft_config=lora_config,

#     # PASS THE COLLATOR HERE
#     data_collator=collator,
# )

## Loop

In [ ]:
trainer.train()

In [ ]:
ADAPTER_DIR = "./qlora-peft-output/adapter"

# 6. Save adapter weights

In [ ]:
os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to: {ADAPTER_DIR}")

In [ ]:
!tree


# Load merge model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, AutoPeftModelForCausalLM

# -------------------------------------------------------
# Paths
# -------------------------------------------------------
BASE_ID = "arnir0/Tiny-LLM"
BASE_ID = "Qwen/Qwen2-1.5B-Instruct"
ADAPTER_DIR = "./qlora-peft-output/adapter"
MERGED_DIR = "./merged-model"

# -------------------------------------------------------
# Load tokenizer
# -------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(
    BASE_ID,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# -------------------------------------------------------
# Load base model in fp16/bf16
# (merged model will be full precision LLM)
# -------------------------------------------------------
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# -------------------------------------------------------
# Attach adapter
# -------------------------------------------------------
print("Loading adapter onto base model...")
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

# -------------------------------------------------------
# Merge adapter → base model
# -------------------------------------------------------
print("Merging LoRA adapter into base model weights...")
merged_model = model.merge_and_unload()   # <-- key line

# -------------------------------------------------------
# Save merged full model (optional)
# -------------------------------------------------------
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print(f"\nMerged model saved to: {MERGED_DIR}\n")

In [ ]:
!tree

# Evaluation

In [ ]:
def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(merged_model.device)
    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
print("==================== MERGED MODEL OUTPUT ====================\n")

for p in test_dataset:
    wrapped = (
        "You are a personalized assistant that knows details about the user based "
        "on prior fine-tuning data.\n\n"
        f"Question: {p['instruction']}\nAnswer:"
    )

    print(f"Output: {p['output']}\n")
    output = generate(wrapped)
    print(output)
    print("\n", "-" * 120, "\n")

In [ ]:
trainer.evaluate()

In [ ]:
metrics = trainer.evaluate()
ppl = math.exp(metrics['eval_loss']) if metrics['eval_loss'] < 20 else float('inf')
print('KL-SFT post-train eval_loss:', metrics['eval_loss'], 'ppl:', ppl)
print('Eval losses over steps:', [(h.get('step'), h.get('eval_loss')) for h in trainer.state.log_history if 'eval_loss' in h])


# Resources

- Wandb
  - https://wandb.ai/capecape/alpaca_ft/reports/How-to-Fine-tune-an-LLM-Part-3-The-HuggingFace-Trainer--Vmlldzo1OTEyNjMy
  - https://youtu.be/LQvRhQwDOm0
- LoRA
  - https://youtu.be/t1caDsMzWBk
- SFT
  - https://youtu.be/cayFaWkI39A
  - https://youtu.be/XOlyOWE4YaM
  - https://huggingface.co/docs/trl/en/sft_trainer